**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/zaki-2024"], check=True)
    os.chdir("/content/encore/entries/zaki-2024")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Earlier findings of zaki-2024-01, checked on version 0.261122.2159

This notebook belongs to the reconcile step of attempt zaki-2024-02 (docs/protocol.md, "Repeat
attempts"). It checks the findings of the earlier attempt, zaki-2024-01 (made on DANDI:000718
version 0.260825.1902), that zaki-2024-02 did not report, on the version zaki-2024-02 pins,
0.261122.2159. The files are streamed with `zaki.py`, the entry's helper.

Earlier findings 1, 2, 4, 6, 7, 8 and 10 are reported again by zaki-2024-02 (its findings 3, 3, 1,
2, 4, 9 and 8) and are not checked here. Earlier finding 9 is reported in part (its finding 5, the
shock amplitude without a unit); its other parts are checked here.

In [ ]:
import numpy as np
import requests
import zaki

print("dandiset", zaki.DANDISET, "version", zaki.VERSION)

## Earlier finding 3

> The dandiset does not say why these two mice, and only the first offline recording of each
> period, were deposited, nor that the paper analysed more.

The dandiset's description and release notes at this version, and the experiment description of
an offline file, are printed, with the sessions the version holds.

In [ ]:
meta = requests.get(f"{zaki.API}/dandisets/{zaki.DANDISET}/versions/{zaki.VERSION}/", timeout=60).json()
paths = sorted(p for p in zaki.assets() if p.endswith(".nwb"))
print("NWB files:", len(paths))
for p in paths:
    print("  ", p.split("/")[1])
print()
print("description:", meta["description"])
print()
print("release notes:")
print(meta.get("releaseNotes"))
f = zaki.open_h5("sub-Ca-EEG2-1/sub-Ca-EEG2-1_ses-OfflineDay2Session1_ophys.nwb")
print()
print("experiment_description:", zaki.text(f["general/experiment_description"]))
words = ("why", "select", "chosen", "subset", "request")
for name, t in [("description", meta["description"]), ("release notes", meta.get("releaseNotes") or ""),
                ("experiment_description", zaki.text(f["general/experiment_description"]))]:
    print(name, "mentions:", [w for w in words if w in t.lower()])

The description is the paper's abstract, and the experiment description says nothing of which
mice or recordings were deposited. The release notes of this version list the two mice and the
first 10 min recording of each offline period under "Known gaps", which says that more was
recorded, but nothing printed above says why these mice and recordings were deposited or which
recordings the paper analysed.

## Earlier finding 5

> The OnePhotonSeries timestamps of three files (Ca_EEG2-1 NeutralExposure, Ca_EEG3-4
> OfflineDay1Session1, Ca_EEG3-4 Recall2) run at 16.7 ms and end at half the recording's length,
> while their frame counts fit 30 Hz over the full length given by the deconvolved timestamps.

For each of the three files, the number of frames, the OnePhotonSeries timestamps (first, last,
median step) and the deconvolved timestamps are printed.

In [ ]:
three = ["sub-Ca-EEG2-1/sub-Ca-EEG2-1_ses-NeutralExposure_image+ophys.nwb",
         "sub-Ca-EEG3-4/sub-Ca-EEG3-4_ses-OfflineDay1Session1_ophys.nwb",
         "sub-Ca-EEG3-4/sub-Ca-EEG3-4_ses-Recall2_image+ophys.nwb"]
for p in three:
    f = zaki.open_h5(p)
    ops = f["acquisition/OnePhotonSeries"]
    n = ops["data"].shape[0]
    t = ops["timestamps"][:]
    d = f["processing/ophys/Fluorescence/Deconvolved/timestamps"][:]
    print(p.split("/")[1])
    print(f"   OnePhotonSeries: {n} frames, {len(t)} timestamps, {t[0]:.3f} to {t[-1]:.3f} s, "
          f"median step {1000 * np.median(np.diff(t)):.1f} ms, rate {(n - 1) / (t[-1] - t[0]):.2f} Hz")
    print(f"   Deconvolved: {len(d)} samples, {d[0]:.3f} to {d[-1]:.3f} s; "
          f"last frame time over last deconvolved time {t[-1] / d[-1]:.3f}")

In all three files the frame times now run over the whole recording, one per frame, ending
0.034 s after the last deconvolved sample, at 29.62 to 29.79 Hz (median step 34.0 ms), not at
16.7 ms over half of it.

## Earlier finding 9

> The fluorescence series carry placeholder descriptions ("description of deconvolved traces")
> and unit "n.a.", the FC session descriptions give the shock amplitude without a unit, and the
> Week file has an empty session description.

The description and unit of each fluorescence series in every file with imaging, the shock
sentence of both FC session descriptions, and the Week file's session description are printed.

In [ ]:
seen = {}
for p in paths:
    f = zaki.open_h5(p)
    if "processing/ophys/Fluorescence" not in f:
        continue
    for k, g in f["processing/ophys/Fluorescence"].items():
        key = (k, str(g.attrs.get("description")), str(g["data"].attrs.get("unit")))
        seen.setdefault(key, []).append(p.split("/")[1].split("_ses-")[1].split("_")[0])
for (k, desc, unit), files in sorted(seen.items()):
    print(f"{k}: description {desc!r}, unit {unit!r}, in {len(files)} files")
for p in ["sub-Ca-EEG2-1/sub-Ca-EEG2-1_ses-FC_image+ophys.nwb", "sub-Ca-EEG3-4/sub-Ca-EEG3-4_ses-FC_image+ophys.nwb"]:
    s = zaki.text(zaki.open_h5(p)["session_description"])
    print(p.split("/")[1], "...", s[s.find("received"):s.find(", with")])
w = zaki.open_h5("sub-Ca-EEG2-1/sub-Ca-EEG2-1_ses-Week.nwb")
print("Week session_description:", repr(zaki.text(w["session_description"])))

All three parts are still there at this version: the four fluorescence series of the 11 files
with imaging keep their placeholder descriptions and unit "n.a.", the FC descriptions give "1.5"
and "0.25" without a unit (zaki-2024-02's finding 5), and the Week file's session description is
empty.